[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/00_what_is_mujoco.ipynb)

# Tutorial 0: What is MuJoCo?

**No prior physics simulation experience needed.**

This tutorial takes you from zero to understanding what MuJoCo does, why it exists, and how to think about it — all through hands-on examples you can see.

In [ ]:
!pip install -q mujoco mediapy

In [ ]:
import mujoco
import numpy as np
import mediapy as media
print(f"MuJoCo {mujoco.__version__} ready!")

---
## What Problem Does MuJoCo Solve?

Imagine you're building a robot. Before building the physical robot (expensive, slow, breakable), you want to:

1. **Test** if your robot design can actually walk/grasp/fly
2. **Train** an AI controller using millions of trial-and-error attempts
3. **Predict** what happens when the robot interacts with objects

MuJoCo is a **physics simulator** — software that computes how objects move under forces, gravity, and contact. It answers: *"If I apply this force, what happens next?"*

### Why MuJoCo specifically?

| Feature | Why it matters |
|---------|---------------|
| **Fast** | 10-100× faster than real-time — train robots in minutes, not years |
| **Accurate contacts** | Handles touching, sliding, bouncing realistically |
| **Stable** | Doesn't "explode" or produce garbage even with complex robots |
| **GPU-parallel (MJX)** | Simulate 10,000 robots simultaneously on one GPU |
| **Free & open-source** | Apache 2.0 license, maintained by Google DeepMind |

---
## The Big Idea: Describe → Simulate → Observe

Using MuJoCo always follows three steps:

```
1. DESCRIBE    →    2. SIMULATE    →    3. OBSERVE
   (what exists)       (advance time)       (read results)
```

Let's see each step with the simplest possible example: **a ball falling under gravity.**

### Step 1: DESCRIBE — Tell MuJoCo what exists

We write a description in XML (a text format called MJCF). Think of it like a recipe:

- "There's a flat ground"
- "There's a ball at height 2 meters"
- "Gravity pulls things down at 9.81 m/s²"

In [ ]:
# This XML string describes our world
world_description = """
<mujoco>
  <option gravity="0 0 -9.81"/>          <!-- Gravity points down (negative Z) -->
  <worldbody>                              <!-- Everything physical goes here -->
    <light pos="0 0 3" dir="0 0 -1"/>    <!-- A light so we can see -->
    <geom type="plane" size="2 2 0.1" rgba=".8 .8 .8 1"/>  <!-- The ground -->
    <body pos="0 0 2">                    <!-- A body (movable thing) at height 2m -->
      <joint type="free"/>                <!-- It can move freely in all directions -->
      <geom type="sphere" size="0.15" rgba="1 0.3 0 1"/>  <!-- It looks like an orange ball -->
    </body>
  </worldbody>
</mujoco>
"""

# MuJoCo "compiles" the description into an efficient internal format
model = mujoco.MjModel.from_xml_string(world_description)
print("✓ World described and compiled!")
print(f"  - Bodies: {model.nbody} (includes the 'world' itself)")
print(f"  - Geoms (shapes): {model.ngeom}")
print(f"  - Gravity: {model.opt.gravity}")

### Step 2: SIMULATE — Advance time

Now we create the **state** (positions, velocities at a moment in time) and ask MuJoCo to compute what happens next.

In [ ]:
# Create the initial state (ball at height 2, not moving)
data = mujoco.MjData(model)

print(f"At time {data.time:.3f}s: ball height = {data.qpos[2]:.3f}m")

# Advance simulation by one tiny timestep (0.002 seconds by default)
mujoco.mj_step(model, data)

print(f"At time {data.time:.3f}s: ball height = {data.qpos[2]:.6f}m")
print(f"  (It moved {2.0 - data.qpos[2]:.6f}m downward — gravity is working!)")

### Step 3: OBSERVE — See what happened

Let's simulate for longer and watch the ball fall and bounce:

In [ ]:
# Reset everything to the start
data = mujoco.MjData(model)

# Create a renderer (a virtual camera)
renderer = mujoco.Renderer(model, height=400, width=500)

# Simulate for 3 seconds, capturing frames for a video
frames = []
while data.time < 3.0:
    mujoco.mj_step(model, data)       # Physics step
    
    # Capture a frame every 1/30th of a second (30 fps video)
    if len(frames) < data.time * 30:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

renderer.close()
media.show_video(frames, fps=30)

**That's it!** You just:
1. Described a world (XML) → compiled to `model`
2. Simulated it (`mj_step` in a loop) → updated `data`
3. Observed the result (rendered frames)

Everything else in MuJoCo builds on this pattern.

---
## The Two Core Objects

MuJoCo has just **two** main data structures. Understanding them is the key to everything:

### `MjModel` — The Blueprint (doesn't change)

Contains everything that stays **constant** during simulation:
- How many objects exist, their shapes and sizes
- Mass, friction, stiffness
- Joint types and limits
- Gravity, timestep settings

**Analogy:** The architectural plan of a building — it tells you what's there, but not where the people are right now.

### `MjData` — The Snapshot (changes every step)

Contains everything that **changes** over time:
- Current positions (`data.qpos`) and velocities (`data.qvel`)
- Current time (`data.time`)
- Forces, contacts, sensor readings
- Control signals (`data.ctrl`)

**Analogy:** A security camera snapshot — tells you where everything is *right now*.

In [ ]:
# model = the blueprint (constant)
print("=== MODEL (blueprint) ===")
print(f"  Timestep: {model.opt.timestep}s (fixed)")
print(f"  Ball radius: {model.geom_size[1, 0]}m (fixed)")
print(f"  Gravity: {model.opt.gravity} (fixed)")

print()

# data = the current state (changes)
print("=== DATA (current snapshot) ===")
print(f"  Time: {data.time:.3f}s")
print(f"  Ball position: {data.qpos[:3]}")
print(f"  Ball velocity: {data.qvel[:3]}")

---
## Building Blocks: Bodies, Joints, Geoms

Every MuJoCo world is made of three building blocks:

| Concept | What it is | Real-world analogy |
|---------|-----------|-------------------|
| **Body** | A rigid frame that can move | A bone in your skeleton |
| **Joint** | Connects bodies, allows specific movement | A knee or elbow |
| **Geom** | A shape attached to a body (for collision & visuals) | The flesh/skin on a bone |

Let's build something step by step and see how they work together:

In [ ]:
# A pendulum: one body, connected to the world by a hinge joint
pendulum_xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 -1 2" dir="0 0.5 -1"/>
    
    <!-- The pendulum arm -->
    <body name="arm" pos="0 0 1.5">
      <!-- THIS is what makes it swing: a hinge joint -->
      <!-- Without this joint, the arm would be frozen in place -->
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      
      <!-- The visible/physical shape -->
      <geom type="capsule" fromto="0 0 0 0 0 -0.8" size="0.04" 
            mass="1" rgba="0.2 0.6 0.9 1"/>
    </body>
  </worldbody>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(pendulum_xml)
d = mujoco.MjData(m)

# Start the pendulum at 90 degrees
d.qpos[0] = np.pi / 2  # 90° in radians

# Simulate
renderer = mujoco.Renderer(m, height=400, width=400)
frames = []
while d.time < 5.0:
    mujoco.mj_step(m, d)
    if len(frames) < d.time * 30:
        renderer.update_scene(d)
        frames.append(renderer.render().copy())
renderer.close()

media.show_video(frames, fps=30)

**What happened:**
- We created one **body** (the arm)
- We gave it a **hinge joint** — this allows rotation around one axis (like a door hinge)
- We gave it a **geom** (capsule shape) — so it has mass and is visible
- Gravity did the rest!

### Joint Types

| Joint type | Degrees of freedom | Example |
|-----------|-------------------|--------|
| `free` | 6 (move + rotate anywhere) | Drone, thrown ball |
| `hinge` | 1 (rotate around axis) | Door, elbow, wheel |
| `slide` | 1 (translate along axis) | Drawer, piston |
| `ball` | 3 (rotate in any direction) | Shoulder, hip |

---
## Adding Control: Making Things Move on Purpose

So far, only gravity moves things. To **control** a robot, we need **actuators** — motors that apply forces to joints.

In [ ]:
controlled_xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 -1 2" dir="0 0.5 -1"/>
    <body pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.8" size="0.04" mass="1" rgba="0.9 0.4 0.1 1"/>
    </body>
  </worldbody>
  
  <!-- NEW: A motor that can apply torque to the hinge -->
  <actuator>
    <motor joint="hinge" ctrlrange="-5 5"/>  <!-- ±5 Nm of torque -->
  </actuator>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(controlled_xml)
d = mujoco.MjData(m)

# Our goal: hold the pendulum at 45 degrees (π/4 radians)
target = np.pi / 4

renderer = mujoco.Renderer(m, height=400, width=400)
frames = []

while d.time < 6.0:
    # === THE CONTROLLER ===
    # Read current angle and speed
    current_angle = d.qpos[0]
    current_speed = d.qvel[0]
    
    # Compute the motor command (PD controller):
    #   - Push toward target (proportional)
    #   - Slow down when moving fast (damping)
    error = target - current_angle
    motor_command = 10.0 * error - 3.0 * current_speed
    
    # Send command to the motor (clipped to ±5 Nm)
    d.ctrl[0] = np.clip(motor_command, -5, 5)
    # === END CONTROLLER ===
    
    mujoco.mj_step(m, d)
    
    if len(frames) < d.time * 30:
        renderer.update_scene(d)
        frames.append(renderer.render().copy())

renderer.close()
media.show_video(frames, fps=30)
print(f"Final angle: {np.degrees(d.qpos[0]):.1f}° (target was {np.degrees(target):.1f}°)")

**The control loop pattern** — this is how ALL robot control works in MuJoCo:

```python
while simulating:
    # 1. READ: Get current state from data
    angle = data.qpos[0]
    
    # 2. DECIDE: Compute what force to apply
    command = my_controller(angle, target)
    
    # 3. ACT: Set the control signal
    data.ctrl[0] = command
    
    # 4. STEP: Let physics happen
    mujoco.mj_step(model, data)
```

---
## Understanding `qpos` and `qvel` (Generalized Coordinates)

You'll see `data.qpos` and `data.qvel` everywhere. Here's the intuition:

- **`qpos`** = all positions/angles that describe the configuration
- **`qvel`** = all velocities (how fast each position is changing)

For a **hinge joint**: `qpos` has 1 number (the angle in radians)  
For a **free joint**: `qpos` has 7 numbers (x, y, z + quaternion rotation)

| Joint | qpos entries | qvel entries |
|-------|-------------|-------------|
| hinge | 1 (angle) | 1 (angular velocity) |
| slide | 1 (position) | 1 (linear velocity) |
| free | 7 (pos + quaternion) | 6 (linear vel + angular vel) |
| ball | 4 (quaternion) | 3 (angular velocity) |

⚠️ **Note:** For `free` and `ball` joints, `qpos` and `qvel` have *different* sizes! This is because rotations need 4 numbers (quaternion) to represent, but only 3 numbers to describe the rate of change.

In [ ]:
# Example: a model with mixed joint types
mixed_xml = """
<mujoco>
  <worldbody>
    <body name="floating_box" pos="0 0 1">
      <joint type="free"/>       <!-- 7 qpos, 6 qvel -->
      <geom type="box" size="0.1 0.1 0.1"/>
    </body>
    <body name="hinged_arm" pos="1 0 1">
      <joint type="hinge" axis="0 1 0"/>  <!-- 1 qpos, 1 qvel -->
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.02"/>
    </body>
  </worldbody>
</mujoco>
"""
m = mujoco.MjModel.from_xml_string(mixed_xml)
d = mujoco.MjData(m)

print(f"Total qpos size: {m.nq} (7 from free + 1 from hinge = 8)")
print(f"Total qvel size: {m.nv} (6 from free + 1 from hinge = 7)")
print(f"")
print(f"qpos = {d.qpos}")  # [x,y,z, qw,qx,qy,qz, hinge_angle]
print(f"qvel = {d.qvel}")  # [vx,vy,vz, wx,wy,wz, hinge_velocity]")

---
## Putting It All Together: A Mini Robot

Let's build a 2-link arm from scratch — combining everything we've learned:

In [ ]:
robot_xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 -1 2" dir="0 0.5 -1"/>
    <geom type="plane" size="1 1 0.1" rgba=".85 .85 .85 1"/>
    
    <!-- Base is fixed to the world -->
    <body name="upper_arm" pos="0 0 0.8">
      <joint name="shoulder" type="hinge" axis="0 1 0"/>     <!-- Shoulder -->
      <geom type="capsule" fromto="0 0 0 0.3 0 0" size="0.03" rgba="0.2 0.5 0.8 1"/>
      
      <body name="forearm" pos="0.3 0 0">
        <joint name="elbow" type="hinge" axis="0 1 0"/>     <!-- Elbow -->
        <geom type="capsule" fromto="0 0 0 0.25 0 0" size="0.025" rgba="0.8 0.4 0.2 1"/>
        <site name="hand" pos="0.25 0 0" size="0.02" rgba="1 0 0 1"/>  <!-- Fingertip -->
      </body>
    </body>
  </worldbody>
  
  <actuator>
    <motor name="shoulder_motor" joint="shoulder" ctrlrange="-10 10"/>
    <motor name="elbow_motor" joint="elbow" ctrlrange="-5 5"/>
  </actuator>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(robot_xml)
d = mujoco.MjData(m)

print(f"Our robot has:")
print(f"  {m.njnt} joints (shoulder + elbow)")
print(f"  {m.nu} actuators (one motor per joint)")
print(f"  {m.nq} degrees of freedom")

# Make the arm trace a circle!
renderer = mujoco.Renderer(m, height=400, width=500)
frames = []

while d.time < 8.0:
    # Move joints in a sinusoidal pattern
    t = d.time
    target_shoulder = 0.5 * np.sin(t * 1.5)
    target_elbow = 0.8 * np.sin(t * 2.0 + 1.0)
    
    # PD control for both joints
    d.ctrl[0] = 20 * (target_shoulder - d.qpos[0]) - 5 * d.qvel[0]
    d.ctrl[1] = 10 * (target_elbow - d.qpos[1]) - 3 * d.qvel[1]
    
    mujoco.mj_step(m, d)
    if len(frames) < d.time * 30:
        renderer.update_scene(d)
        frames.append(renderer.render().copy())

renderer.close()
media.show_video(frames, fps=30)

---
## Summary: The MuJoCo Mental Model

```
┌─────────────────────────────────────────────────────────┐
│                    YOUR CODE                             │
│                                                         │
│  1. Write XML (once)  ──►  model = MjModel(xml)        │
│                                    │                    │
│  2. Create state      ──►  data = MjData(model)        │
│                                    │                    │
│  3. Loop:                          ▼                    │
│     ┌──────────────────────────────────────┐            │
│     │  Read: angle = data.qpos[0]          │            │
│     │  Decide: command = controller(angle)  │            │
│     │  Act: data.ctrl[0] = command          │            │
│     │  Step: mj_step(model, data)           │ ◄── repeat│
│     │  (Optional) Render: renderer.render() │            │
│     └──────────────────────────────────────┘            │
└─────────────────────────────────────────────────────────┘
```

### Key vocabulary

| Term | Meaning |
|------|--------|
| **MJCF** | The XML format for describing models |
| **MjModel** | Compiled model (constant blueprint) |
| **MjData** | Simulation state (changes each step) |
| **qpos** | Generalized positions (angles/positions of all joints) |
| **qvel** | Generalized velocities |
| **ctrl** | Control signals sent to actuators |
| **mj_step** | Advance physics by one timestep |
| **mj_forward** | Recompute derived quantities without advancing time |

---
## What's Next?

Now that you understand the fundamentals, continue with:

1. **[Tutorial 01: Basics](01_basics.ipynb)** — Named access, model inspection
2. **[Tutorial 02: Rendering](02_rendering.ipynb)** — Cameras, depth, segmentation
3. **[Tutorial 03: Simulation](03_simulation.ipynb)** — Timesteps, energy, integrators
4. **[Tutorial 04: Control](04_control.ipynb)** — PD control, servos, gain tuning

Or jump to specific topics:
- Building robots from XML → [Tutorial 06](06_xml_modeling.ipynb)
- Reinforcement Learning → [Tutorial 08](08_gymnasium.ipynb)
- GPU-parallel simulation → [Tutorial 07](07_mjx.ipynb)